In [1]:
from pathlib import Path

import duckdb
import mmlib

import pandas as pd

In [2]:
ROOT_PATH = Path(".").resolve().absolute()
GRAPHHOPPER_BASE_URL = "http://localhost:8989"
GRAPHHOPPER_GPS_ACCURACY = 30 # meters
VEHICLE_ID = 5
SAMPLE_RATE: int | None = 2  # seconds

In [3]:
GROUND_TRUTH_PATH = ROOT_PATH / "sumo/simulations/ohare-chicago/output/fcd.parquet"
NOISE_PARQUET = ROOT_PATH / "sumo/simulations/ohare-chicago/output/fcd_noisy.parquet"

In [4]:
type Coordinate = tuple[float, float]  # (latitude, longitude)

In [5]:

def load_road_dataset(path: Path) -> pd.DataFrame:
    df = duckdb.query(
        f"""                
        SELECT lon, lat, TO_TIMESTAMP(time) AS timestamp, edge_id::STRING AS edge_id
        FROM '{path}'
        WHERE vehicle_id = {VEHICLE_ID}
        ORDER BY time ASC
        """
    ).to_df()

    if SAMPLE_RATE is not None:
        df = df.resample(f"{SAMPLE_RATE:.2f}s", on="timestamp").first().reset_index()
    
    return df


In [6]:
gt_df = load_road_dataset(GROUND_TRUTH_PATH)
gt_df

,timestamp,lon,lat,edge_id
0,1969-12-31 21:00:18-03:00,-87.904743,41.981548,1027375129
1,1969-12-31 21:00:20-03:00,-87.904780,41.981537,1027375129
2,1969-12-31 21:00:22-03:00,-87.904889,41.981474,1027375129
3,1969-12-31 21:00:24-03:00,-87.905103,41.981397,1027375129
4,1969-12-31 21:00:26-03:00,-87.905312,41.981263,1027375129
...,...,...,...,...
216,1969-12-31 21:07:30-03:00,-87.884164,41.994854,1094426129
217,1969-12-31 21:07:32-03:00,-87.883892,41.994854,1094426129
218,1969-12-31 21:07:34-03:00,-87.883629,41.994835,9288515121
219,1969-12-31 21:07:36-03:00,-87.883362,41.994837,450282724


In [7]:
noisy_df = load_road_dataset(NOISE_PARQUET)
noisy_df

,timestamp,lon,lat,edge_id
0,1969-12-31 21:00:18-03:00,-87.904766,41.981579,1027375129
1,1969-12-31 21:00:20-03:00,-87.904702,41.981523,1027375129
2,1969-12-31 21:00:22-03:00,-87.904928,41.981354,1027375129
3,1969-12-31 21:00:24-03:00,-87.905096,41.981477,1027375129
4,1969-12-31 21:00:26-03:00,-87.905286,41.981224,1027375129
...,...,...,...,...
216,1969-12-31 21:07:30-03:00,-87.884157,41.994819,1094426129
217,1969-12-31 21:07:32-03:00,-87.883833,41.994718,1094426129
218,1969-12-31 21:07:34-03:00,-87.883565,41.994896,9288515121
219,1969-12-31 21:07:36-03:00,-87.883379,41.994843,450282724


In [8]:
gt_gps_coordinates = gt_df[["lat", "lon", "timestamp"]].to_records(index=False).tolist()
gt_points = gt_df[["lat", "lon"]].to_records(index=False).tolist()
gt_edge_ids = gt_df["edge_id"].tolist()

print("GROUND TRUTH: ", len(gt_points), "points")
print("first 3 points:", gt_points[:3])
print("first 3 edge ids:", gt_edge_ids[:3])
print("fist 3 coordinates:", gt_gps_coordinates[:3])

GROUND TRUTH:  221 points
first 3 points: [(41.981548227395244, -87.9047432890212), (41.981537308888214, -87.90478027380234), (41.981474167273156, -87.90488912350818)]
first 3 edge ids: ['1027375129', '1027375129', '1027375129']
fist 3 coordinates: [(41.981548227395244, -87.9047432890212, Timestamp('1969-12-31 21:00:18-0300', tz='America/Fortaleza')), (41.981537308888214, -87.90478027380234, Timestamp('1969-12-31 21:00:20-0300', tz='America/Fortaleza')), (41.981474167273156, -87.90488912350818, Timestamp('1969-12-31 21:00:22-0300', tz='America/Fortaleza'))]


In [9]:
noisy_gps_coordinates = noisy_df[["lat", "lon", "timestamp"]].to_records(index=False).tolist()
noisy_points = noisy_df[["lat", "lon"]].to_records(index=False).tolist()
noisy_edge_ids = noisy_df["edge_id"].tolist()

print("NOISY: ", len(noisy_points), "points")
print("first 3 points:", noisy_points[:3])
print("first 3 edge ids:", noisy_edge_ids[:3])
print("first 3 coordinates:", noisy_gps_coordinates[:3])

NOISY:  221 points
first 3 points: [(41.98157884752087, -87.90476632005051), (41.98152344886095, -87.90470188871046), (41.981353749912145, -87.90492772098426)]
first 3 edge ids: ['1027375129', '1027375129', '1027375129']
first 3 coordinates: [(41.98157884752087, -87.90476632005051, Timestamp('1969-12-31 21:00:18-0300', tz='America/Fortaleza')), (41.98152344886095, -87.90470188871046, Timestamp('1969-12-31 21:00:20-0300', tz='America/Fortaleza')), (41.981353749912145, -87.90492772098426, Timestamp('1969-12-31 21:00:22-0300', tz='America/Fortaleza'))]


In [10]:
matcher = mmlib.graphhopper_matcher(GRAPHHOPPER_BASE_URL, gps_accuracy=GRAPHHOPPER_GPS_ACCURACY)

In [11]:
gpx_data = mmlib.to_gpx(noisy_gps_coordinates)

In [12]:
match_result = matcher.map_match(gpx_data)
match_result

MatchResult(points=[Coordinate(latitude=41.98154, longitude=-87.90462), Coordinate(latitude=41.98142, longitude=-87.90503), Coordinate(latitude=41.98136, longitude=-87.90514), Coordinate(latitude=41.98126, longitude=-87.90529), Coordinate(latitude=41.9812, longitude=-87.90536), Coordinate(latitude=41.98113, longitude=-87.90542), Coordinate(latitude=41.98103, longitude=-87.90548), Coordinate(latitude=41.9809, longitude=-87.90553), Coordinate(latitude=41.9806, longitude=-87.90559), Coordinate(latitude=41.98037, longitude=-87.90564), Coordinate(latitude=41.98015, longitude=-87.90573), Coordinate(latitude=41.97911, longitude=-87.90597), Coordinate(latitude=41.97872, longitude=-87.90592), Coordinate(latitude=41.9783, longitude=-87.90602), Coordinate(latitude=41.97797, longitude=-87.90607), Coordinate(latitude=41.97783, longitude=-87.90606), Coordinate(latitude=41.97774, longitude=-87.90604), Coordinate(latitude=41.97763, longitude=-87.90596), Coordinate(latitude=41.97752, longitude=-87.9058

In [28]:
ground_truth_edges = set(gt_edge_ids)
matched_edges = set(match_result.edge_ids)
(
    len(matched_edges), 
    len(ground_truth_edges), len(matched_edges & ground_truth_edges),
    len(matched_edges & ground_truth_edges), 
    len(matched_edges.symmetric_difference(ground_truth_edges))
)

(41, 56, 34, 34, 29)

In [14]:
from plotly import graph_objects as go


def plot_trajectories(
    original: list[Coordinate],
    calculated: list[Coordinate],
    title: str = "Tracks",
    original_label: str = "Original",
    calculated_label: str = "Calculated",
    show_original_line: bool = True,
    show_buttons: bool = True,
    center_lat: float | None = None,
    center_lon: float | None = None,
    zoom: float = 14,
) -> None:
    flat_original = [
        (lat, lon, original_label, i) for i, (lat, lon) in enumerate(original)
    ]
    flat_calculated = [
        (lat, lon, calculated_label, i) for i, (lat, lon) in enumerate(calculated)
    ]

    df = pd.DataFrame(
        flat_original + flat_calculated,
        columns=["lat", "lon", "type", "row_num"],
    )

    fig = go.Figure()
    visible_flags = []

    for type_ in [original_label, calculated_label]:
        subset = df[df["type"] == type_].sort_values("row_num")

        # Pontos
        fig.add_trace(
            go.Scattermap(
                lat=subset["lat"],
                lon=subset["lon"],
                mode="markers",
                marker=dict(size=10),
                name=f"{type_} - pontos",
                legendgroup=type_,
                visible=True,
            )
        )
        visible_flags.append(True)

        # Linhas
        show_line = True
        if type_ == original_label and not show_original_line:
            show_line = False
        fig.add_trace(
            go.Scattermap(
                lat=subset["lat"],
                lon=subset["lon"],
                mode="lines",
                line=dict(width=2),
                name=f"{type_} - linha",
                legendgroup=type_,
                visible=show_line,
            )
        )
        visible_flags.append(show_line)

    # Definir centro do mapa
    if center_lat is None or center_lon is None:
        # Usar centro calculado a partir dos dados
        map_center = dict(lat=df["lat"].mean(), lon=df["lon"].mean())
    else:
        # Usar coordenadas fornecidas
        map_center = dict(lat=center_lat, lon=center_lon)

    if show_buttons:
        fig.update_layout(
            updatemenus=[
                dict(
                    type="buttons",
                    direction="right",
                    showactive=True,
                    x=0.5,
                    xanchor="center",
                    y=1,
                    yanchor="top",
                    buttons=[
                        dict(
                            label="Mostrar Nenhuma",
                            method="update",
                            args=[
                                {"visible": [False, False, False, False]},
                                {"mapbox": dict(center=map_center, zoom=zoom)},
                            ],
                        ),
                        dict(
                            label="Mostrar Ambas",
                            method="update",
                            args=[
                                {"visible": [True, show_original_line, True, True]},
                                {"mapbox": dict(center=map_center, zoom=zoom)},
                            ],
                        ),
                        dict(
                            label=f"Apenas {original_label}",
                            method="update",
                            args=[
                                {"visible": [True, show_original_line, False, False]},
                                {"mapbox": dict(center=map_center, zoom=zoom)},
                            ],
                        ),
                        dict(
                            label=f"Apenas {calculated_label}",
                            method="update",
                            args=[
                                {"visible": [False, False, True, True]},
                                {"mapbox": dict(center=map_center, zoom=zoom)},
                            ],
                        ),
                    ],
                )
            ]
        )

    fig.update_layout(
        mapbox_style="open-street-map",
        mapbox=dict(center=map_center, zoom=zoom),
        margin=dict(l=0, r=0, t=80, b=0),
        height=700,
        title=title,
    )

    fig.show()

In [15]:
plot_trajectories(
    original=noisy_points,
    calculated=[pt.to_tuple() for pt in match_result.points],
    title="Map Matching - GraphHopper",
    original_label="Noisy GPS",
    calculated_label="GraphHopper Match",
    show_original_line=False,
    zoom=16,
)